In [2]:
%%writefile app.py
import streamlit as st
import requests
import pandas as pd
import sqlite3
import hashlib
from datetime import datetime
from io import BytesIO
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from streamlit_option_menu import option_menu
import plotly.express as px

# ==============================================================================
# PAGE CONFIG & MULTILINGUAL DICTIONARY
# ==============================================================================
st.set_page_config(
    page_title="Social Media Impact Predictor",
    page_icon="🧠",
    layout="wide"
)

# Language Dictionary for Translation
lang_dict = {
    "English": {
        "title": "🧠 Social Media Impact & Mental Health Predictor",
        "subtitle": "This dashboard uses **Machine Learning** to predict mental health risk and **Gemini AI** for digital well-being consultation.",
        "demo_head": "📊 Demographics & Usage",
        "behavior_head": "📉 Behavioral Metrics (Rate 1 to 5)",
        "btn_run": "🔥 Run AI & ML Diagnosis",
        "nav_title": "🧭 Navigation",
        "menu_diag": "🤖 User Diagnosis & AI",
        "menu_chat": "💬 Live AI Chatbot Buddy",
        "menu_goals": "🎯 Daily Goal Tracker",
        "menu_hist": "📜 My Consultation History",
        "menu_model": "📊 Model Insights & Performance"
    },
    "Urdu": {
        "title": "🧠 سوشل میڈیا اثرات اور ذہنی صحت کا تجزیہ",
        "subtitle": "یہ ڈیش بورڈ ذہنی صحت کے خطرے کی پیش گوئی کرنے کے لیے **مشین لرننگ** اور ڈیجیٹل فلاح و بہبود کے مشورے کے لیے **جیمنی اے آئی** کا استعمال کرتا ہے۔",
        "demo_head": "📊 آبادیاتی معلومات اور استعمال",
        "behavior_head": "📉 رویے کے میٹرکس (ریٹنگ 1 سے 5)",
        "btn_run": "🔥 اے آئی اور ایم ایل تشخیص چلائیں",
        "nav_title": "🧭 نیویگیشن",
        "menu_diag": "🤖 صارف کی تشخیص اور اے آئی",
        "menu_chat": "💬 لائیو اے آئی چیٹ بوٹ دوست",
        "menu_goals": "🎯 روزانہ کے اہداف کا ٹریکر",
        "menu_hist": "📜 میری پرانی ہسٹری لاگز",
        "menu_model": "📊 ماڈل کی کارکردگی اور معلومات"
    }
}

# Language Selector in Sidebar Top
st.sidebar.markdown("### 🌐 Language / زبان")
selected_lang = st.sidebar.selectbox("Choose Language:", ["English", "Urdu"], label_visibility="collapsed")
t = lang_dict[selected_lang]

# ==============================================================================
# DATABASE FUNCTIONS (SQLite)
# ==============================================================================
def init_db():
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    c.execute("CREATE TABLE IF NOT EXISTS users (username TEXT PRIMARY KEY, password TEXT)")
    c.execute("""
        CREATE TABLE IF NOT EXISTS history (
            id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT, date TEXT, 
            risk_level TEXT, impact_score REAL, ai_consult TEXT
        )
    """)
    c.execute("CREATE TABLE IF NOT EXISTS goals (username TEXT, goal_text TEXT, completed INTEGER, date TEXT)")
    conn.commit()
    conn.close()

def make_hashes(password): return hashlib.sha256(str.encode(password)).hexdigest()
def check_hashes(password, hashed_text): return make_hashes(password) == hashed_text

def add_user(username, password):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    try:
        c.execute("INSERT INTO users(username, password) VALUES (?,?)", (username, make_hashes(password)))
        conn.commit(); success = True
    except sqlite3.IntegrityError: success = False
    conn.close(); return success

def login_user(username, password):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    c.execute("SELECT password FROM users WHERE username = ?", (username,))
    data = c.fetchone(); conn.close()
    if data: return check_hashes(password, data[0])
    return False

def save_history(username, risk, score, ai_text):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    c.execute("INSERT INTO history(username, date, risk_level, impact_score, ai_consult) VALUES (?,?,?,?,?)",
              (username, datetime.now().strftime("%Y-%m-%d %H:%M:%S"), risk, score, ai_text))
    conn.commit(); conn.close()

def get_user_history(username):
    conn = sqlite3.connect("users.db")
    df = pd.read_sql_query("SELECT date, risk_level, impact_score, ai_consult FROM history WHERE username = ? ORDER BY date DESC", conn, params=(username,))
    conn.close(); return df

def get_goals(username):
    conn = sqlite3.connect("users.db")
    df = pd.read_sql_query("SELECT rowid, goal_text, completed FROM goals WHERE username = ?", conn, params=(username,))
    conn.close(); return df

def add_goal(username, text):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    c.execute("INSERT INTO goals VALUES (?, ?, 0, ?)", (username, text, datetime.now().strftime("%Y-%m-%d")))
    conn.commit(); conn.close()

def toggle_goal(rowid, status):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    c.execute("UPDATE goals SET completed = ? WHERE rowid = ?", (status, rowid))
    conn.commit(); conn.close()

init_db()

# ==============================================================================
# PDF GENERATION FUNCTION
# ==============================================================================
def generate_pdf_report(username, risk, score, consultation):
    buffer = BytesIO()
    doc = SimpleDocTemplate(buffer, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []
    
    title_style = ParagraphStyle('TitleStyle', parent=styles['Heading1'], fontSize=22, spaceAfter=20)
    body_style = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontSize=11, leading=16, spaceAfter=12)
    
    story.append(Paragraph("<b>🧠 Social Media Impact & Mental Health Report</b>", title_style))
    story.append(Paragraph(f"<b>User:</b> {username}", body_style))
    story.append(Paragraph(f"<b>Date:</b> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", body_style))
    story.append(Spacer(1, 10))
    story.append(Paragraph(f"<b>Predicted Mental Health Risk Level:</b> {risk}", body_style))
    story.append(Paragraph(f"<b>Overall Social Media Impact Score:</b> {score} / 5", body_style))
    story.append(Spacer(1, 15))
    story.append(Paragraph("<b>🤖 Gemini AI Personalized Consultation Summary:</b>", styles['Heading2']))
    story.append(Paragraph(consultation.replace("\n", "<br/>"), body_style))
    
    doc.build(story)
    buffer.seek(0)
    return buffer

# ==============================================================================
# AUTHENTICATION & SESSION MANAGEMENT
# ==============================================================================
if "logged_in" not in st.session_state: st.session_state["logged_in"] = False
if "username" not in st.session_state: st.session_state["username"] = ""
if "messages" not in st.session_state: st.session_state["messages"] = []

API_URL = "http://127.0.0.1:8000/predict_and_consult"
CHAT_API_URL = "http://127.0.0.1:8000/chat"

if not st.session_state["logged_in"]:
    st.title("🔒 Welcome - Social Media Mental Health Portal")
    auth_mode = st.tabs(["🔑 Login", "📝 Sign Up"])
    with auth_mode[0]:
        l_user = st.text_input("Username", key="l_user")
        l_pass = st.text_input("Password", type="password", key="l_pass")
        if st.button("Log In", type="primary"):
            if login_user(l_user, l_pass):
                st.session_state["logged_in"] = True
                st.session_state["username"] = l_user
                st.rerun()
            else: st.error("Invalid credentials.")
    with auth_mode[1]:
        r_user = st.text_input("Choose Username", key="r_user")
        r_pass = st.text_input("Choose Password", type="password", key="r_pass")
        if st.button("Register Account"):
            if r_user and r_pass and add_user(r_user, r_pass): st.success("Account created! Go to Login.")
            else: st.error("Username exists or fields are empty.")
else:
    # Main Navigation Sidebar (Updated with option_menu)
    with st.sidebar:
        st.markdown(f"### 👤 Welcome,\n## {st.session_state['username']}")
        st.markdown("---")
        
        # Professional Navigation Menu
        page = option_menu(
            menu_title=t["nav_title"],  
            options=[t["menu_diag"], t["menu_chat"], t["menu_goals"], t["menu_hist"], t["menu_model"]],
            icons=["clipboard2-pulse", "chat-dots", "bullseye", "clock-history", "graph-up-arrow"],
            menu_icon="cast",
            default_index=0,
            styles={
                "container": {"padding": "0!important", "background-color": "transparent"},
                "icon": {"color": "#00d2ff", "font-size": "18px"},
                "nav-link": {"font-size": "16px", "text-align": "left", "margin": "0px", "--hover-color": "#262730"},
                "nav-link-selected": {"background-color": "#00d2ff", "color": "#0e1117", "font-weight": "bold"},
            }
        )
        st.markdown("---")
        if st.button("🚪 Log Out", use_container_width=True):
            st.session_state["logged_in"] = False; st.session_state["username"] = ""; st.rerun()

    # --- 1. USER DIAGNOSIS PAGE ---
    if page == t["menu_diag"]:
        st.title(t["title"])
        st.markdown(f"*{t['subtitle']}*")
        st.divider() # Professional separator
        
        st.subheader(t["demo_head"])
        # Grid layout for inputs (FIXED spacing with gap="large")
        col1, col2, col3 = st.columns(3, gap="large")
        with col1:
            age = st.number_input("Age", min_value=10, max_value=80, value=22)
            gender = st.selectbox("Gender", options=[("Male", 1), ("Female", 0), ("Other", 2)], format_func=lambda x: x[0])[1]
        with col2:
            relationship = st.selectbox("Relationship Status", options=[("Single", 0), ("In a Relationship", 1), ("Married", 2)], format_func=lambda x: x[0])[1]
            occupation = st.selectbox("Occupation", options=[("Student", 1), ("Salaried", 2), ("Self-Employed", 3), ("Unemployed", 0)], format_func=lambda x: x[0])[1]
        with col3:
            org_type = st.selectbox("Organization Type", options=[("None", 0), ("University", 1), ("Corporate", 2), ("Government", 3)], format_func=lambda x: x[0])[1]
            platforms = st.number_input("Platforms Used Actively", min_value=0, max_value=10, value=3)
        
        st.markdown("<br>", unsafe_allow_html=True) # Spacing
        
        st.subheader(t["behavior_head"])
        # Two columns for sliders (FIXED overlapping)
        b_col1, b_col2 = st.columns(2, gap="large")
        with b_col1:
            usage_hours = st.slider("Daily Usage (Hours)", 0.0, 24.0, 4.5, step=0.5)
            st.markdown("<br>", unsafe_allow_html=True)
            purposeless = st.slider("Scroll purposelessly?", 1, 5, 3)
            st.markdown("<br>", unsafe_allow_html=True)
            distracted = st.slider("Distracted by notifications?", 1, 5, 4)
            st.markdown("<br>", unsafe_allow_html=True)
            restless = st.slider("Restless without phone?", 1, 5, 2)
            st.markdown("<br>", unsafe_allow_html=True)
            concentration = st.slider("Concentration level (1=Poor, 5=Excellent)", 1, 5, 3)
        with b_col2:
            focus_issue = st.slider("Frequency of focus issues?", 1, 5, 4)
            st.markdown("<br>", unsafe_allow_html=True)
            comparison = st.slider("Compare yourself to others?", 1, 5, 3)
            st.markdown("<br>", unsafe_allow_html=True)
            comparison_feelings = st.slider("Badly affect feelings?", 1, 5, 2)
            st.markdown("<br>", unsafe_allow_html=True)
            validation = st.slider("Seek validation (likes/comments)?", 1, 5, 4)
            st.markdown("<br>", unsafe_allow_html=True)
            interest_fluctuation = st.slider("Interest in daily tasks fluctuate?", 1, 5, 3)

        st.markdown("<br>", unsafe_allow_html=True)

        if st.button(t["btn_run"], type="primary", use_container_width=True):
            payload = {
                "age": age, "gender": gender, "relationship": relationship, "occupation": occupation,
                "org_type": org_type, "platforms": platforms, "usage_hours": usage_hours,
                "purposeless": purposeless, "distracted": distracted, "restless": restless,
                "concentration": concentration, "focus_issue": focus_issue, "comparison": comparison,
                "comparison_feelings": comparison_feelings, "validation": validation, "interest_fluctuation": interest_fluctuation
            }
            with st.spinner("Processing through ML & Gemini AI..."):
                try:
                    response = requests.post(API_URL, json=payload)
                    if response.status_code == 200:
                        result = response.json()
                        res_col1, res_col2 = st.columns([1, 2], gap="large")
                        risk = result["ml_results"]["predicted_risk"]
                        score = result["ml_results"]["overall_score"]
                        ai_consultation = result["ai_consultation"]
                        
                        save_history(st.session_state["username"], risk, score, ai_consultation)
                        
                        with res_col1:
                            st.success("### 🤖 ML Model Output")
                            if risk == "Low": 
                                st.metric(label="Risk Level", value=risk, delta="Safe")
                            elif risk == "Medium": 
                                st.metric(label="Risk Level", value=risk, delta="Warning", delta_color="off")
                            else: 
                                st.metric(label="Risk Level", value=risk, delta="Critical", delta_color="inverse")
                            st.metric(label="Overall Impact Score", value=f"{score} / 5")
                            
                            # Feature Action: PDF Generation Button
                            pdf_data = generate_pdf_report(st.session_state["username"], risk, score, ai_consultation)
                            st.download_button(label="📄 Download PDF Report", data=pdf_data, file_name=f"Mental_Health_Report_{st.session_state['username']}.pdf", mime="application/pdf", use_container_width=True)
                        with res_col2:
                            st.info("### ✨ Gemini AI Consultation")
                            st.write(ai_consultation)
                    else: st.error("Backend server error.")
                except Exception as e: st.error("Connection Error: Server is not running.")

    # --- 2. LIVE AI CHATBOT BUDDY ---
    elif page == t["menu_chat"]:
        st.title("💬 Live AI Mental Health Buddy")
        st.markdown("Feel free to discuss anything about digital anxiety, screen time management, or social media pressure.")
        st.markdown("---")
        
        for msg in st.session_state["messages"]:
            with st.chat_message(msg["role"]): st.write(msg["content"])
            
        user_msg = st.chat_input("Type your message here...")
        if user_msg:
            st.session_state["messages"].append({"role": "user", "content": user_msg})
            with st.chat_message("user"): st.write(user_msg)
            
            with st.spinner("Buddy is thinking..."):
                try:
                    chat_res = requests.post(CHAT_API_URL, json={"message": user_msg, "language": selected_lang})
                    if chat_res.status_code == 200:
                        bot_reply = chat_res.json()["response"]
                        st.session_state["messages"].append({"role": "assistant", "content": bot_reply})
                        with st.chat_message("assistant"): st.write(bot_reply)
                    else: st.error("Chatbot backend error.")
                except Exception as e: st.error(f"Error: {str(e)}")

    # --- 3. DAILY GOAL TRACKER ---
    elif page == t["menu_goals"]:
        st.title("🎯 Habit & Well-being Goal Tracker")
        st.markdown("Build healthy digital habits by setting and checking off positive goals daily.")
        st.markdown("---")
        
        g_col1, g_col2 = st.columns([1, 2])
        with g_col1:
            st.subheader("Add New Goal")
            new_goal_txt = st.text_input("Example: Limit screen time to 2 hours today", key="new_goal")
            if st.button("Add Goal", type="primary", use_container_width=True):
                if new_goal_txt:
                    add_goal(st.session_state["username"], new_goal_txt)
                    st.success("Goal Added!")
                    st.rerun()
        with g_col2:
            st.subheader("Your Active Targets")
            goals_df = get_goals(st.session_state["username"])
            if goals_df.empty: st.info("No goals set yet! Put your targets on the left.")
            else:
                for idx, row in goals_df.iterrows():
                    is_done = bool(row["completed"])
                    # Checkbox interaction directly changing DB status
                    checked = st.checkbox(row["goal_text"], value=is_done, key=f"goal_{row['rowid']}")
                    if checked != is_done:
                        toggle_goal(row["rowid"], 1 if checked else 0)
                        st.rerun()

    # --- 4. HISTORY LOGS ---
    elif page == t["menu_hist"]:
        st.title(t["menu_hist"])
        history_df = get_user_history(st.session_state["username"])
        if history_df.empty: st.info("No records found.")
        else:
            h_col1, h_col2 = st.columns(2)
            with h_col1:
                st.subheader("Trends Chart")
                chart_data = history_df.sort_values(by="date").set_index("date")[["impact_score"]]
                st.line_chart(chart_data)
            with h_col2:
                st.subheader("Summary")
                st.dataframe(history_df[["date", "risk_level", "impact_score"]], use_container_width=True)
            st.markdown("---")
            selected_date = st.selectbox("Select Timestamp:", history_df["date"].tolist())
            if selected_date:
                selected_row = history_df[history_df["date"] == selected_date].iloc[0]
                st.warning(f"Risk: {selected_row['risk_level']} | Score: {selected_row['impact_score']}/5")
                st.info(selected_row["ai_consult"])

    # --- 5. MODEL INSIGHTS (Updated with Plotly) ---
    elif page == t["menu_model"]:
        st.title(t["menu_model"])
        st.markdown("Detailed breakdown of machine learning algorithms evaluated during model training.")
        
        model_data = {
            "Model": ["Logistic Regression", "Random Forest", "XGBoost", "Gradient Boosting", "Decision Tree", "SVM"],
            "Test Accuracy": [0.7952, 0.7831, 0.7711, 0.7590, 0.7108, 0.6265],
            "CV Mean": [0.7518, 0.7325, 0.7084, 0.7301, 0.6867, 0.6169]
        }
        df_models = pd.DataFrame(model_data)
        
        st.dataframe(df_models.style.highlight_max(axis=0, color='#00d2ff'), use_container_width=True)
        
        # Professional Plotly Bar Chart
        st.subheader("Accuracy Comparison")
        fig = px.bar(df_models, x='Model', y='Test Accuracy', 
                     color='Test Accuracy', 
                     color_continuous_scale='Blues',
                     title="Model Test Accuracies")
        fig.update_layout(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)")
        st.plotly_chart(fig, use_container_width=True)

Overwriting app.py
